In [1]:
# @title
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random
import glob
from scipy.ndimage import gaussian_filter, zoom
import pickle

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/IRP/scripts')

from SegNet import SegNet, SegNetDataset
from plot_predictions import plot_all
from CentreNetStrided import SCDCentreNet, CentreNetDataset

In [3]:
# @title
def set_seed(seed):
    """
    Use this to set ALL the random seeds to a fixed value and take out any randomness from cuda kernels
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False  ##uses the inbuilt cudnn auto-tuner to find the fastest convolution algorithms. -
    torch.backends.cudnn.enabled   = False

    return True

device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")

No GPU available!


In [4]:
root = Path('/content/drive/MyDrive/IRP/SNIPApproved')
paths = [root / 'Brazil/pos_g0030_474830_8185045_477902_8188117.npz',
         root / 'Russia2/pos_g0003_286186_5638657_291306_5643777.npz',
         root / 'Russia/neg_226265_5694233_229337_5697305.npz',
         root / 'Australia/pos_g0000_400399_6614557_403471_6617629.npz']

# make dataset/loader for selected examples tiles
seg_dset = SegNetDataset(paths, augment=False)
centre_dset = CentreNetDataset(paths, augment=False)
seg_loader = DataLoader(seg_dset, batch_size=4, shuffle=False, num_workers=0)
centre_loader = DataLoader(centre_dset, batch_size=4, shuffle=False, num_workers=0)

Found 4 tiles (0 partial, 0 missing bands skipped)
Found 4 tiles (0 partial, 0 missing bands skipped)


Instantiate Models

In [7]:
segnet = SegNet(in_channels=4, base_filters=64)
centrenet = SCDCentreNet(in_channels=4, base_filters=64)

segnet.load_state_dict(torch.load('/content/drive/MyDrive/IRP/models/seg_nocv_75.pkl', weights_only=True))
centrenet.load_state_dict(torch.load('/content/drive/MyDrive/models/cn_nocv_75.pkl', weights_only=True))

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL SegNet.SegNet was not an allowed global by default. Please use `torch.serialization.add_safe_globals([SegNet.SegNet])` or the `torch.serialization.safe_globals([SegNet.SegNet])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weight

Get Predictions

In [ ]:
plot_all(segnet, centrenet, seg_loader, centre_loader, device)